In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
# ---------- 1. Данные ----------
df = pd.read_csv('/Users/taniyashuba/PycharmProjects/NeuroSymbolicDynamics/data/lorenz.csv')
series = df['x'].values.astype(np.float32)

seq_len = 10
def make_dataset(data, seq_len):
    X, Y = [], []
    for i in range(len(data)-seq_len):
        X.append(data[i:i+seq_len])
        Y.append(data[i+seq_len])
    X = np.stack(X)    # (N, seq_len)
    Y = np.stack(Y)    # (N,)
    return X[...,None], Y[...,None]

X, Y = make_dataset(series, seq_len)
split = 4000
X_train, Y_train = X[:split], Y[:split]
X_val,   Y_val   = X[split:split+500], Y[split:split+500]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
X_train = torch.tensor(X_train).to(device)  # (N, seq_len, 1)
Y_train = torch.tensor(Y_train).to(device)
X_val   = torch.tensor(X_val).to(device)
Y_val   = torch.tensor(Y_val).to(device)

In [3]:
# ---------- 2. Модель Transformer ----------
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.pe = pe.unsqueeze(1)  # (max_len,1,d_model)
    def forward(self, x):
        # x: (seq_len, batch, d_model)
        x = x + self.pe[:x.size(0)]
        return x

class DecoderOnlyTransformer(nn.Module):
    def __init__(self, d_model=32, nhead=4, nlayers=2, dim_feedforward=128):
        super().__init__()
        self.input_proj = nn.Linear(1, d_model)
        self.pos_enc = PositionalEncoding(d_model)
        dec_layer = nn.TransformerDecoderLayer(d_model, nhead, dim_feedforward)
        self.decoder = nn.TransformerDecoder(dec_layer, nlayers)
        self.out = nn.Linear(d_model, 1)

    def forward(self, src):
        # src: (batch, seq_len, 1) -> (seq_len, batch, 1)
        src = src.transpose(0,1)
        x = self.input_proj(src)             # (seq_len, batch, d_model)
        x = self.pos_enc(x)                  # + positional encoding

        # формируем маску, чтобы блокировать будущее
        seq_len = x.size(0)
        mask = torch.triu(torch.ones(seq_len, seq_len, device=x.device) * float('-inf'),
                           diagonal=1)

        # НЕ ПЕРЕДАЁМ memory=None, а подаём x как memory:
        out = self.decoder(tgt=x, memory=x, tgt_mask=mask)
        # out: (seq_len, batch, d_model)

        pred = self.out(out[-1])             # последний шаг для предсказания
        return pred, out

# Instantiation
model = DecoderOnlyTransformer().to(device)
criterion = nn.MSELoss()
opt = optim.Adam(model.parameters(), lr=1e-3)

In [4]:
# ---------- 3. Training ----------
epochs = 50
for ep in range(1, epochs+1):
    model.train()
    opt.zero_grad()
    pred, _ = model(X_train)
    loss = criterion(pred, Y_train)
    loss.backward()
    opt.step()
    if ep%10==0:
        model.eval()
        with torch.no_grad():
            val_pred, _ = model(X_val)
            val_loss = criterion(val_pred, Y_val)
        print(f"Epoch {ep:02d}  train_loss={loss.item():.5f}  val_loss={val_loss.item():.5f}")

Epoch 10  train_loss=34.23704  val_loss=34.09095
Epoch 20  train_loss=28.54886  val_loss=29.38393
Epoch 30  train_loss=25.97463  val_loss=26.97651
Epoch 40  train_loss=23.54002  val_loss=24.71391
Epoch 50  train_loss=21.18043  val_loss=22.56207


In [5]:
# ---------- 4. Съём скрытых состояний ----------
model.eval()
with torch.no_grad():
    _, traj = model(X_val)      # traj: (seq_len, batch, d_model)
    # хотим массив N_val × seq_len × d_model
    hidden_trf = traj.transpose(0,1).cpu().numpy()

In [6]:
# Сохраняем в файл
np.save('hidden_trf.npy', hidden_trf)   # shape (N_val, seq_len, d_model)
print("Saved Transformer hidden states:", hidden_trf.shape)

Saved Transformer hidden states: (500, 10, 32)


In [7]:
# Загружаем .npy-файл
hidden_states = np.load('/Users/taniyashuba/PycharmProjects/NeuroSymbolicDynamics/hidden_trf.npy')

# Проверяем тип данных и форму массива
print(f"Тип данных: {type(hidden_states)}")
print(f"Форма массива: {hidden_states.shape}")
print(f"Пример содержимого:\n{hidden_states[0]}")

Тип данных: <class 'numpy.ndarray'>
Форма массива: (500, 10, 32)
Пример содержимого:
[[-0.2001867  -0.25167555 -1.6388097   1.5898231  -0.8892281  -0.22057824
   0.24824202  1.6894946   0.15075901  1.3467133  -2.3270009   0.3460329
   0.12966111  1.0153599  -0.84801644  1.4471654  -0.39949784  2.1115723
  -0.93425417  0.57272553 -0.97073185 -0.9760887  -0.35458326 -0.13336374
  -1.2905691   0.22633836 -0.70174193  0.09491917 -0.42976016  0.63059425
   0.05471651  0.9338142 ]
 [ 0.36596268 -0.45572078 -1.4564321   1.5745546  -0.9142549  -0.08785252
   0.27964437  1.5612701   0.08410816  1.357944   -2.1489513   0.22582136
   0.09644078  0.9288576  -0.88274693  1.6945013  -0.53815126  1.9971061
  -0.8314416   0.6760984  -1.1809593  -0.92983675 -0.45407486 -0.19774203
  -1.5863765   0.22674023 -0.69182     0.24618264 -0.5154009   0.6476456
   0.05765695  0.8740915 ]
 [ 0.3553225  -0.9748215  -1.1903721   1.2301154  -0.6085179  -0.27276292
   0.44375044  1.5495902   0.24798313  1.2808473  -